# 🛡️ DeepMineRisk: Intelligent Mining Process Monitoring and Risk Prediction using Deep Learning

**B.E. AIML Final Year Project**

This notebook builds an end-to-end AI system that:

- Loads and cleans real mining flotation process sensor data
- Performs exploratory data analysis (EDA)
- Engineers useful time-series features
- Trains an **LSTM** deep learning model to predict process/quality conditions
- Converts predictions into an **AI Risk Score (0–100)** with Safe / Warning / Critical levels
- Generates automatic **recommendations** based on risk
- Explains predictions using **SHAP**
- Saves all artifacts and generates a full **Streamlit dashboard** (`app.py`) plus deployment files

> **Instructions:** Just run every cell from top to bottom. When prompted, upload the dataset ZIP file (`MiningProcess_Flotation_Plant_Database.zip` containing the CSV). No manual code edits are required.


## 1. Environment Setup

Install/verify all required libraries. This is optimized for the default Google Colab T4 runtime.

In [ ]:
# Install required packages (Colab usually already has most of these)
!pip install -q tensorflow scikit-learn plotly shap streamlit pyngrok kaleido --upgrade
print("✅ Packages installed successfully.")

## 2. Import Libraries

In [ ]:
import os
import io
import re
import json
import glob
import shutil
import zipfile
import pickle
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 3. Upload & Extract Dataset

Upload the Kaggle ZIP file containing `MiningProcess_Flotation_Plant_Database.csv`. The cell below automatically extracts the ZIP and detects the CSV file — no need to type file names manually.

In [ ]:
from google.colab import files

PROJECT_DIR = "/content/DeepMineRisk"
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
os.makedirs(DATASET_DIR, exist_ok=True)

print("📂 Please upload the dataset ZIP file (e.g. MiningProcess_Flotation_Plant_Database.zip)")
uploaded = files.upload()

zip_path = None
for fname in uploaded.keys():
    if fname.lower().endswith(".zip"):
        zip_path = fname
        break

if zip_path is None:
    raise FileNotFoundError("No ZIP file detected in the upload. Please upload a .zip file.")

print(f"✅ ZIP file received: {zip_path}")

In [ ]:
# Automatically extract the ZIP file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)

print(f"✅ Extracted contents to: {DATASET_DIR}")

# Automatically detect the CSV file inside the extracted folder (recursive search)
csv_candidates = glob.glob(os.path.join(DATASET_DIR, "**", "*.csv"), recursive=True)

if len(csv_candidates) == 0:
    raise FileNotFoundError("No CSV file found inside the uploaded ZIP.")

# Prefer a file that matches the expected mining dataset name, else take the first CSV
csv_path = None
for c in csv_candidates:
    if "mining" in os.path.basename(c).lower():
        csv_path = c
        break
if csv_path is None:
    csv_path = csv_candidates[0]

print(f"✅ Detected CSV file: {csv_path}")

## 4. Load Dataset

In [ ]:
# Load CSV (try comma decimal separator first, since Kaggle version uses commas as decimals)
try:
    df_raw = pd.read_csv(csv_path, decimal=",")
except Exception as e:
    print("Falling back to default decimal parsing:", e)
    df_raw = pd.read_csv(csv_path)

print(f"Shape: {df_raw.shape}")
df_raw.head()

## 5. Data Cleaning

Automatically:
- Convert decimal-comma strings (e.g. `55,2` → `55.2`) to proper floats
- Convert every numeric-like column to `float`
- Convert the date column to `datetime`
- Handle missing values
- Remove duplicate rows
- Verify final data types

In [ ]:
df = df_raw.copy()

def comma_to_float(series):
    """Convert a column of strings like '55,2' into float 55.2. Leaves already-numeric columns untouched."""
    if series.dtype == object:
        cleaned = series.astype(str).str.replace(",", ".", regex=False).str.strip()
        converted = pd.to_numeric(cleaned, errors="coerce")
        # Only accept conversion if it doesn't wipe out all data
        if converted.notna().sum() > 0:
            return converted
    return series

# Identify the date column automatically (case-insensitive match on 'date')
date_col = None
for col in df.columns:
    if "date" in col.lower():
        date_col = col
        break

for col in df.columns:
    if col == date_col:
        continue
    df[col] = comma_to_float(df[col])

# Convert date column to datetime
if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.sort_values(date_col).reset_index(drop=True)

print("✅ Decimal-comma conversion and dtype casting complete.")
df.dtypes

In [ ]:
# Handle missing values: forward-fill then back-fill (good for time-series sensor data),
# then drop any fully-empty rows that remain
missing_before = df.isnull().sum().sum()

df = df.ffill().bfill()
df = df.dropna(how="all")

missing_after = df.isnull().sum().sum()
print(f"Missing values before: {missing_before} | after: {missing_after}")

# Remove duplicate rows
dupes_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {dupes_before}")

print(f"✅ Final cleaned dataset shape: {df.shape}")
df.head()

In [ ]:
# Final data type check
print("Data types after cleaning:")
print(df.dtypes)
print("\nSummary statistics:")
df.describe().T

## 6. Exploratory Data Analysis (EDA)

Professional interactive visualizations built with Plotly to understand the sensor data.

### 6.1 Dataset Overview

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print(f"Numeric Columns: {len(numeric_cols)}")
if date_col:
    print(f"Date Range: {df[date_col].min()} → {df[date_col].max()}")

fig = go.Figure(data=[go.Table(
    header=dict(values=["Metric", "Value"], fill_color="#1f2937", font=dict(color="white"), align="left"),
    cells=dict(values=[
        ["Rows", "Columns", "Numeric Columns", "Duplicate Rows"],
        [df.shape[0], df.shape[1], len(numeric_cols), df.duplicated().sum()]
    ], align="left"))
])
fig.update_layout(title="Dataset Overview", height=300)
fig.show()

### 6.2 Missing Values

In [ ]:
missing_df = df.isnull().sum().reset_index()
missing_df.columns = ["Column", "MissingCount"]
missing_df = missing_df[missing_df["MissingCount"] > 0]

if missing_df.empty:
    print("✅ No missing values remain in the dataset.")
else:
    fig = px.bar(missing_df, x="Column", y="MissingCount", title="Missing Values per Column",
                 color="MissingCount", color_continuous_scale="Reds")
    fig.show()

### 6.3 Correlation Heatmap

In [ ]:
corr = df[numeric_cols].corr()

fig = px.imshow(corr, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r",
                 title="Correlation Heatmap of Sensor Features")
fig.update_layout(height=800)
fig.show()

### 6.4 Target Distribution

We predict **`% Silica Concentrate`** when available (a key quality/impurity indicator that is expensive to measure directly and drives the risk score). If not present, the last numeric column is used automatically.

In [ ]:
TARGET_COL = None
for col in df.columns:
    if "silica concentrate" in col.lower():
        TARGET_COL = col
        break
if TARGET_COL is None:
    TARGET_COL = numeric_cols[-1]

print(f"🎯 Target column selected: {TARGET_COL}")

fig = px.histogram(df, x=TARGET_COL, nbins=50, marginal="box",
                    title=f"Distribution of Target: {TARGET_COL}", color_discrete_sequence=["#2563eb"])
fig.show()

### 6.5 Sensor Trend Over Time

In [ ]:
trend_col = TARGET_COL
if date_col:
    fig = px.line(df, x=date_col, y=trend_col, title=f"{trend_col} Trend Over Time")
else:
    fig = px.line(df, y=trend_col, title=f"{trend_col} Trend (Index Order)")
fig.show()

### 6.6 Boxplot of Key Sensor Features

In [ ]:
box_cols = numeric_cols[:8]  # first 8 numeric columns to keep the plot readable
fig = px.box(df[box_cols], title="Boxplot of Key Sensor Features (Outlier Check)")
fig.show()

### 6.7 Pairwise Correlation with Target

In [ ]:
target_corr = corr[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False)

fig = px.bar(target_corr, orientation="h", title=f"Feature Correlation with {TARGET_COL}",
             color=target_corr.values, color_continuous_scale="RdBu_r")
fig.update_layout(yaxis_title="Feature", xaxis_title="Correlation")
fig.show()

## 7. Feature Engineering

Automatically engineered time-series features that help the LSTM model capture process dynamics:

- Average Air Flow / Average Column Level (if flotation column sensors exist)
- Feed Ratio
- Rolling Mean / Rolling Std
- Lag Features
- Moving Average
- Difference Features

In [ ]:
df_feat = df.copy()

air_flow_cols = [c for c in df_feat.columns if "air flow" in c.lower()]
level_cols = [c for c in df_feat.columns if "level" in c.lower()]
iron_feed_cols = [c for c in df_feat.columns if "iron feed" in c.lower()]
silica_feed_cols = [c for c in df_feat.columns if "silica feed" in c.lower()]

# Average Air Flow across flotation columns
if air_flow_cols:
    df_feat["Avg_Air_Flow"] = df_feat[air_flow_cols].mean(axis=1)

# Average Column Level across flotation columns
if level_cols:
    df_feat["Avg_Column_Level"] = df_feat[level_cols].mean(axis=1)

# Feed Ratio: Iron Feed / Silica Feed (guard against divide-by-zero)
if iron_feed_cols and silica_feed_cols:
    df_feat["Feed_Ratio"] = df_feat[iron_feed_cols[0]] / (df_feat[silica_feed_cols[0]] + 1e-6)

# Rolling statistics, lag, moving average and difference features on the target column
window = 5
df_feat[f"{TARGET_COL}_RollingMean"] = df_feat[TARGET_COL].rolling(window=window, min_periods=1).mean()
df_feat[f"{TARGET_COL}_RollingStd"] = df_feat[TARGET_COL].rolling(window=window, min_periods=1).std().fillna(0)
df_feat[f"{TARGET_COL}_Lag1"] = df_feat[TARGET_COL].shift(1)
df_feat[f"{TARGET_COL}_MovingAvg10"] = df_feat[TARGET_COL].rolling(window=10, min_periods=1).mean()
df_feat[f"{TARGET_COL}_Diff"] = df_feat[TARGET_COL].diff()

# Fill any NaNs created by shifting/differencing
df_feat = df_feat.ffill().bfill()

print(f"✅ Feature engineering complete. New shape: {df_feat.shape}")
new_cols = [c for c in df_feat.columns if c not in df.columns]
print("Engineered features:", new_cols)
df_feat[new_cols].head()

## 8. Data Scaling

Scale all numeric features to the `[0, 1]` range using `MinMaxScaler`, then persist the scaler with `pickle` for reuse in the dashboard.

In [ ]:
feature_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns.tolist()]

scaler = MinMaxScaler()
scaled_array = scaler.fit_transform(df_feat[feature_cols])
df_scaled = pd.DataFrame(scaled_array, columns=feature_cols)

os.makedirs(PROJECT_DIR, exist_ok=True)
with open(os.path.join(PROJECT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(PROJECT_DIR, "features.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)

print(f"✅ Scaled {len(feature_cols)} numeric features and saved scaler.pkl + features.pkl")
df_scaled.head()

## 9. Sequence Creation for LSTM

Build sliding-window sequences (`SEQUENCE_LENGTH = 10`) so the LSTM can learn temporal patterns in the sensor data.

In [ ]:
SEQUENCE_LENGTH = 10
target_idx = feature_cols.index(TARGET_COL)

def create_sequences(data, seq_len, target_index):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len, target_index])
    return np.array(X), np.array(y)

X, y = create_sequences(df_scaled.values, SEQUENCE_LENGTH, target_idx)
print(f"✅ Sequences created → X shape: {X.shape}, y shape: {y.shape}")

## 10. Train / Test Split (80% / 20%)

Chronological split (no shuffling) since this is time-series sensor data.

In [ ]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 11. Deep Learning Model — LSTM

A single, focused LSTM architecture (no BiLSTM / GRU / Transformer variants):

```
Input → LSTM(32) → Dropout → LSTM(16) → Dense(16) → Dense(1)
```

In [ ]:
n_features = X_train.shape[2]

model = keras.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, n_features)),
    layers.LSTM(32, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(16),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
], name="DeepMineRisk_LSTM")

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
model.summary()

### Training Callbacks & Model Training

`EarlyStopping`, `ReduceLROnPlateau`, and `ModelCheckpoint` are used so training automatically stops if validation loss stops improving, and the best model is preserved.

In [ ]:
os.makedirs(PROJECT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(PROJECT_DIR, "DeepMineRisk_Model.keras")

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
checkpoint = ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print(f"✅ Training complete. Best model saved to: {MODEL_PATH}")

## 12. Model Evaluation

Compute MAE, RMSE, R², MAPE, and generate diagnostic plots.

In [ ]:
y_pred = model.predict(X_test).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-6))) * 100

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

### Training & Validation Loss Curves

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history["loss"], name="Training Loss", mode="lines"))
fig.add_trace(go.Scatter(y=history.history["val_loss"], name="Validation Loss", mode="lines"))
fig.update_layout(title="Training vs Validation Loss", xaxis_title="Epoch", yaxis_title="Loss")
fig.show()

### Actual vs Predicted, Prediction Error, and Residual Plots

In [ ]:
# Actual vs Predicted
fig1 = go.Figure()
fig1.add_trace(go.Scatter(y=y_test, name="Actual", mode="lines"))
fig1.add_trace(go.Scatter(y=y_pred, name="Predicted", mode="lines"))
fig1.update_layout(title="Actual vs Predicted (Test Set)", xaxis_title="Sample", yaxis_title=TARGET_COL)
fig1.show()

# Prediction Error
error = y_test - y_pred
fig2 = px.line(error, title="Prediction Error over Test Samples")
fig2.update_layout(yaxis_title="Error", xaxis_title="Sample")
fig2.show()

# Residual Plot
fig3 = px.scatter(x=y_pred, y=error, title="Residual Plot",
                   labels={"x": "Predicted Value", "y": "Residual"})
fig3.add_hline(y=0, line_dash="dash", line_color="red")
fig3.show()

## 13. Intelligent Risk Prediction

Instead of showing a raw predicted value, convert every prediction into an **AI Risk Score (0–100)** and a Risk Level (Safe / Warning / Critical) using min-max normalization against the observed target range.

In [ ]:
target_min = df_feat[TARGET_COL].min()
target_max = df_feat[TARGET_COL].max()

def compute_risk_score(pred_value, tmin=target_min, tmax=target_max):
    """Normalize a raw prediction into a 0-100 AI Risk Score."""
    score = (pred_value - tmin) / (tmax - tmin + 1e-9) * 100
    return float(np.clip(score, 0, 100))

def risk_level(score):
    if score < 40:
        return "Safe", "🟢"
    elif score < 70:
        return "Warning", "🟡"
    else:
        return "Critical", "🔴"

# Demonstrate on a few test predictions (inverse-scaled back to original units for display)
sample_preds = y_pred[:5]
print("Sample AI Risk Scores:\n")
for i, p in enumerate(sample_preds):
    score = compute_risk_score(p)
    level, icon = risk_level(score)
    print(f"Sample {i+1}: Predicted={p:.3f} | Risk Score={score:.1f}/100 | Status: {icon} {level}")

## 14. Recommendation Engine

Automatically generate actionable recommendations based on the predicted risk level.

In [ ]:
def get_recommendation(level):
    recommendations = {
        "Safe": [
            "Continue monitoring",
            "No immediate action required",
            "Maintain current operating parameters"
        ],
        "Warning": [
            "Inspect flotation columns",
            "Check sensor calibration",
            "Reduce feed rate slightly and monitor closely"
        ],
        "Critical": [
            "Increase ventilation immediately",
            "Reduce feed rate",
            "Inspect flotation columns and sensor calibration urgently",
            "Notify shift supervisor"
        ]
    }
    return recommendations.get(level, ["Continue monitoring"])

for i, p in enumerate(sample_preds):
    score = compute_risk_score(p)
    level, icon = risk_level(score)
    recs = get_recommendation(level)
    print(f"Sample {i+1} → {icon} {level} (Score: {score:.1f})")
    for r in recs:
        print(f"   • {r}")
    print()

## 15. Explainable AI (SHAP)

Use SHAP's `DeepExplainer` to interpret LSTM predictions. If `DeepExplainer` is not supported in the current TensorFlow/SHAP version, the notebook gracefully skips this step without stopping execution (KernelExplainer is intentionally avoided since it is too slow for sequence models).

In [ ]:
try:
    import shap

    background = X_train[np.random.choice(X_train.shape[0], min(50, X_train.shape[0]), replace=False)]
    explainer = shap.DeepExplainer(model, background)

    sample_to_explain = X_test[:5]
    shap_values = explainer.shap_values(sample_to_explain)

    print("✅ SHAP DeepExplainer ran successfully.")

    # Average absolute SHAP value per feature (averaged over timesteps) for a simple bar chart
    sv = np.array(shap_values)
    sv = sv.reshape(-1, sv.shape[-2], sv.shape[-1]) if sv.ndim > 3 else sv
    mean_abs_shap = np.mean(np.abs(sv), axis=(0, 1)).flatten()

    n_show = min(len(feature_cols), len(mean_abs_shap))
    shap_df = pd.DataFrame({
        "Feature": feature_cols[:n_show],
        "MeanAbsSHAP": mean_abs_shap[:n_show]
    }).sort_values("MeanAbsSHAP", ascending=False).head(15)

    fig = px.bar(shap_df, x="MeanAbsSHAP", y="Feature", orientation="h",
                 title="SHAP Feature Importance (Mean |SHAP value|)")
    fig.show()

except Exception as e:
    print("⚠️ SHAP DeepExplainer is not supported in this environment. Skipping SHAP explainability gracefully.")
    print(f"   Reason: {e}")

## 16. Save Model & Artifacts

Confirm that `DeepMineRisk_Model.keras`, `scaler.pkl`, and `features.pkl` have all been saved (scaler and features were already saved in the scaling step; the best model was saved automatically via `ModelCheckpoint`).

In [ ]:
saved_files = {
    "Model": os.path.join(PROJECT_DIR, "DeepMineRisk_Model.keras"),
    "Scaler": os.path.join(PROJECT_DIR, "scaler.pkl"),
    "Features": os.path.join(PROJECT_DIR, "features.pkl"),
}

for name, path in saved_files.items():
    status = "✅ Found" if os.path.exists(path) else "❌ Missing"
    print(f"{name}: {path} → {status}")

## 17. Streamlit Dashboard (`app.py`)

A modern, interactive, dark-mode-friendly Streamlit dashboard with sidebar navigation across 6 pages: **Home, Dataset Analytics, Prediction, Risk Dashboard, Recommendations, About Project**.

In [ ]:
%%writefile /content/DeepMineRisk/app.py
import os
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from tensorflow import keras

# ------------------------------------------------------------------
# Page Config
# ------------------------------------------------------------------
st.set_page_config(
    page_title="DeepMineRisk | Mining Risk Monitoring",
    page_icon="⛏️",
    layout="wide",
    initial_sidebar_state="expanded",
)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, "DeepMineRisk_Model.keras")
SCALER_PATH = os.path.join(BASE_DIR, "scaler.pkl")
FEATURES_PATH = os.path.join(BASE_DIR, "features.pkl")
SEQUENCE_LENGTH = 10

# ------------------------------------------------------------------
# Custom CSS (dark-mode friendly)
# ------------------------------------------------------------------
st.markdown("""
<style>
.metric-card {
    background: linear-gradient(135deg, #1f2937, #111827);
    padding: 20px; border-radius: 14px; color: white; text-align: center;
    box-shadow: 0 4px 12px rgba(0,0,0,0.25);
}
.risk-safe { background: linear-gradient(135deg,#16a34a,#166534); padding:18px; border-radius:14px; color:white; }
.risk-warning { background: linear-gradient(135deg,#eab308,#a16207); padding:18px; border-radius:14px; color:white; }
.risk-critical { background: linear-gradient(135deg,#dc2626,#7f1d1d); padding:18px; border-radius:14px; color:white; }
</style>
""", unsafe_allow_html=True)

# ------------------------------------------------------------------
# Cached Loaders
# ------------------------------------------------------------------
@st.cache_resource
def load_model():
    if os.path.exists(MODEL_PATH):
        return keras.models.load_model(MODEL_PATH)
    return None

@st.cache_resource
def load_scaler():
    if os.path.exists(SCALER_PATH):
        with open(SCALER_PATH, "rb") as f:
            return pickle.load(f)
    return None

@st.cache_resource
def load_features():
    if os.path.exists(FEATURES_PATH):
        with open(FEATURES_PATH, "rb") as f:
            return pickle.load(f)
    return None

def risk_level(score):
    if score < 40:
        return "Safe", "🟢", "risk-safe"
    elif score < 70:
        return "Warning", "🟡", "risk-warning"
    else:
        return "Critical", "🔴", "risk-critical"

def get_recommendation(level):
    recs = {
        "Safe": ["Continue monitoring", "No immediate action required", "Maintain current operating parameters"],
        "Warning": ["Inspect flotation columns", "Check sensor calibration", "Reduce feed rate slightly and monitor closely"],
        "Critical": ["Increase ventilation immediately", "Reduce feed rate",
                     "Inspect flotation columns and sensor calibration urgently", "Notify shift supervisor"],
    }
    return recs.get(level, ["Continue monitoring"])

if "prediction_history" not in st.session_state:
    st.session_state.prediction_history = []

# ------------------------------------------------------------------
# Sidebar Navigation
# ------------------------------------------------------------------
st.sidebar.title("⛏️ DeepMineRisk")
st.sidebar.caption("Intelligent Mining Process Monitoring & Risk Prediction")
page = st.sidebar.radio(
    "Navigate",
    ["🏠 Home", "📊 Dataset Analytics", "🔮 Prediction", "🚦 Risk Dashboard", "💡 Recommendations", "ℹ️ About Project"],
)

st.sidebar.markdown("---")
uploaded_file = st.sidebar.file_uploader("Upload Sensor CSV", type=["csv"])

df = None
if uploaded_file is not None:
    try:
        df = pd.read_csv(uploaded_file, decimal=",")
    except Exception:
        uploaded_file.seek(0)
        df = pd.read_csv(uploaded_file)
    st.sidebar.success(f"✅ Loaded {df.shape[0]} rows, {df.shape[1]} columns")

model = load_model()
scaler = load_scaler()
feature_cols = load_features()

# ------------------------------------------------------------------
# HOME PAGE
# ------------------------------------------------------------------
if page == "🏠 Home":
    st.title("⛏️ DeepMineRisk")
    st.subheader("Intelligent Mining Process Monitoring and Risk Prediction using Deep Learning")
    st.markdown("""
    Welcome to **DeepMineRisk** — an AI-powered dashboard that monitors mining flotation process
    sensor data and predicts an intelligent, easy-to-read **Risk Score** in real time.

    **Key Capabilities**
    - 📊 Rich, interactive dataset analytics
    - 🔮 LSTM-powered process prediction
    - 🚦 0–100 AI Risk Score with Safe / Warning / Critical status
    - 💡 Automatic, risk-aware recommendations
    - 📥 Downloadable prediction reports
    """)
    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown('<div class="metric-card"><h3>Model</h3><p>LSTM (32→16 units)</p></div>', unsafe_allow_html=True)
    with col2:
        st.markdown('<div class="metric-card"><h3>Sequence Length</h3><p>10 timesteps</p></div>', unsafe_allow_html=True)
    with col3:
        status = "✅ Loaded" if model is not None else "⚠️ Not Found"
        st.markdown(f'<div class="metric-card"><h3>Model Status</h3><p>{status}</p></div>', unsafe_allow_html=True)

# ------------------------------------------------------------------
# DATASET ANALYTICS PAGE
# ------------------------------------------------------------------
elif page == "📊 Dataset Analytics":
    st.title("📊 Dataset Analytics")
    if df is None:
        st.info("⬅️ Upload a CSV file from the sidebar to view analytics.")
    else:
        with st.spinner("Analyzing dataset..."):
            st.subheader("Dataset Preview")
            st.dataframe(df.head(20), use_container_width=True)

            st.subheader("Dataset Statistics")
            st.dataframe(df.describe().T, use_container_width=True)

            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if numeric_cols:
                st.subheader("Feature Selector")
                selected_feature = st.selectbox("Select a feature to visualize", numeric_cols)

                col1, col2 = st.columns(2)
                with col1:
                    fig = px.histogram(df, x=selected_feature, title=f"Distribution: {selected_feature}", nbins=40)
                    st.plotly_chart(fig, use_container_width=True)
                with col2:
                    fig2 = px.line(df, y=selected_feature, title=f"Trend: {selected_feature}")
                    st.plotly_chart(fig2, use_container_width=True)

                st.subheader("Correlation Heatmap")
                corr = df[numeric_cols].corr()
                fig3 = px.imshow(corr, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r")
                st.plotly_chart(fig3, use_container_width=True)
        st.success("✅ Analysis complete")

# ------------------------------------------------------------------
# PREDICTION PAGE
# ------------------------------------------------------------------
elif page == "🔮 Prediction":
    st.title("🔮 Run Prediction")

    if model is None or scaler is None or feature_cols is None:
        st.error("❌ Model, scaler, or feature list not found. Please ensure DeepMineRisk_Model.keras, "
                  "scaler.pkl, and features.pkl are in the same folder as app.py.")
    elif df is None:
        st.info("⬅️ Upload a CSV file from the sidebar to run predictions.")
    else:
        st.markdown("Date filter lets you focus the prediction window on a specific time range.")
        date_col = next((c for c in df.columns if "date" in c.lower()), None)
        working_df = df.copy()

        if date_col:
            try:
                working_df[date_col] = pd.to_datetime(working_df[date_col], errors="coerce")
                min_d, max_d = working_df[date_col].min(), working_df[date_col].max()
                date_range = st.date_input("Filter by date range", (min_d, max_d))
                if isinstance(date_range, tuple) and len(date_range) == 2:
                    working_df = working_df[(working_df[date_col] >= pd.to_datetime(date_range[0])) &
                                             (working_df[date_col] <= pd.to_datetime(date_range[1]))]
            except Exception:
                pass

        if st.button("🚀 Run Prediction", type="primary"):
            with st.spinner("Running LSTM inference..."):
                try:
                    available = [c for c in feature_cols if c in working_df.columns]
                    missing = [c for c in feature_cols if c not in working_df.columns]
                    for c in missing:
                        working_df[c] = 0.0

                    ordered = working_df[feature_cols].apply(pd.to_numeric, errors="coerce").ffill().bfill().fillna(0)

                    if len(ordered) < SEQUENCE_LENGTH:
                        st.error(f"Need at least {SEQUENCE_LENGTH} rows of data to form a sequence.")
                    else:
                        scaled = scaler.transform(ordered)
                        last_seq = scaled[-SEQUENCE_LENGTH:].reshape(1, SEQUENCE_LENGTH, len(feature_cols))
                        pred_scaled = float(model.predict(last_seq, verbose=0).flatten()[0])

                        target_min, target_max = 0.0, 1.0
                        score = float(np.clip(pred_scaled * 100, 0, 100))
                        level, icon, css_class = risk_level(score)
                        recs = get_recommendation(level)

                        st.session_state.prediction_history.append({
                            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "Prediction": round(pred_scaled, 4),
                            "Risk Score": round(score, 1),
                            "Risk Level": level,
                            "Recommendation": "; ".join(recs),
                        })

                        st.success("✅ Prediction complete!")
                        col1, col2 = st.columns([1, 1])
                        with col1:
                            gauge = go.Figure(go.Indicator(
                                mode="gauge+number",
                                value=score,
                                title={"text": "AI Risk Score"},
                                gauge={
                                    "axis": {"range": [0, 100]},
                                    "bar": {"color": "#111827"},
                                    "steps": [
                                        {"range": [0, 40], "color": "#16a34a"},
                                        {"range": [40, 70], "color": "#eab308"},
                                        {"range": [70, 100], "color": "#dc2626"},
                                    ],
                                },
                            ))
                            st.plotly_chart(gauge, use_container_width=True)
                        with col2:
                            st.markdown(f'<div class="{css_class}"><h2>{icon} {level}</h2>'
                                        f'<p>Risk Score: {score:.1f} / 100</p></div>', unsafe_allow_html=True)
                except Exception as e:
                    st.error(f"❌ Prediction failed: {e}")

        if st.session_state.prediction_history:
            st.subheader("📋 Prediction History")
            hist_df = pd.DataFrame(st.session_state.prediction_history)
            st.dataframe(hist_df, use_container_width=True)

            csv_bytes = hist_df.to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download Prediction Report", data=csv_bytes,
                                file_name="Prediction_Report.csv", mime="text/csv")

# ------------------------------------------------------------------
# RISK DASHBOARD PAGE
# ------------------------------------------------------------------
elif page == "🚦 Risk Dashboard":
    st.title("🚦 Risk Dashboard")
    if not st.session_state.prediction_history:
        st.info("Run a prediction from the Prediction page first to populate the risk dashboard.")
    else:
        hist_df = pd.DataFrame(st.session_state.prediction_history)
        latest = hist_df.iloc[-1]

        col1, col2, col3 = st.columns(3)
        col1.metric("Latest Risk Score", f"{latest['Risk Score']}/100")
        col2.metric("Latest Risk Level", latest["Risk Level"])
        col3.metric("Total Predictions", len(hist_df))

        fig = px.line(hist_df, x="Timestamp", y="Risk Score", markers=True, title="Risk Score Trend")
        fig.add_hline(y=40, line_dash="dash", line_color="green")
        fig.add_hline(y=70, line_dash="dash", line_color="red")
        st.plotly_chart(fig, use_container_width=True)

        level_counts = hist_df["Risk Level"].value_counts().reset_index()
        level_counts.columns = ["Risk Level", "Count"]
        fig2 = px.pie(level_counts, names="Risk Level", values="Count", title="Risk Level Distribution",
                      color="Risk Level",
                      color_discrete_map={"Safe": "#16a34a", "Warning": "#eab308", "Critical": "#dc2626"})
        st.plotly_chart(fig2, use_container_width=True)

# ------------------------------------------------------------------
# RECOMMENDATIONS PAGE
# ------------------------------------------------------------------
elif page == "💡 Recommendations":
    st.title("💡 Recommendations Panel")
    if not st.session_state.prediction_history:
        st.info("Run a prediction first to see tailored recommendations.")
    else:
        latest = st.session_state.prediction_history[-1]
        level = latest["Risk Level"]
        icon = {"Safe": "🟢", "Warning": "🟡", "Critical": "🔴"}.get(level, "⚪")
        st.markdown(f"### Current Status: {icon} {level}")
        for rec in latest["Recommendation"].split("; "):
            st.write(f"- {rec}")

# ------------------------------------------------------------------
# ABOUT PAGE
# ------------------------------------------------------------------
elif page == "ℹ️ About Project":
    st.title("ℹ️ About DeepMineRisk")
    st.markdown("""
    **Project:** DeepMineRisk — Intelligent Mining Process Monitoring and Risk Prediction using Deep Learning

    **Type:** B.E. AIML Final Year Project

    **Model:** Single-layer-stack LSTM (LSTM(32) → Dropout → LSTM(16) → Dense(16) → Dense(1))

    **Pipeline:** Data Cleaning → Feature Engineering → MinMax Scaling → Sequence Creation →
    LSTM Training → Risk Scoring → Recommendation Engine → SHAP Explainability → Streamlit Dashboard

    **Tech Stack:** Python, TensorFlow/Keras, Scikit-learn, Pandas, NumPy, Plotly, Streamlit, SHAP
    """)

st.sidebar.markdown("---")
st.sidebar.caption("© DeepMineRisk — AIML Final Year Project")


## 18. Prediction Report Generation

Generate `Prediction_Report.csv` containing Prediction, Risk Score, Risk Level, Recommendation, and Timestamp for the test set predictions.

In [ ]:
report_rows = []
for i in range(len(y_pred)):
    score = compute_risk_score(y_pred[i])
    level, icon = risk_level(score)
    recs = "; ".join(get_recommendation(level))
    report_rows.append({
        "Prediction": round(float(y_pred[i]), 4),
        "Risk Score": round(score, 1),
        "Risk Level": level,
        "Recommendation": recs,
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

report_df = pd.DataFrame(report_rows)
report_path = os.path.join(PROJECT_DIR, "Prediction_Report.csv")
report_df.to_csv(report_path, index=False)

print(f"✅ Prediction_Report.csv saved with {len(report_df)} rows.")
report_df.head(10)

## 19. Deployment Files

Automatically generate `requirements.txt`, `README.md`, and `.gitignore` for deployment.

In [ ]:
requirements_txt = """tensorflow>=2.15
scikit-learn>=1.3
pandas>=2.0
numpy>=1.24
plotly>=5.18
streamlit>=1.32
shap>=0.44
kaleido>=0.2
"""

with open(os.path.join(PROJECT_DIR, "requirements.txt"), "w") as f:
    f.write(requirements_txt)

print("✅ requirements.txt generated")
print(requirements_txt)

In [ ]:
readme_md = f"""# DeepMineRisk: Intelligent Mining Process Monitoring and Risk Prediction using Deep Learning

## Project Overview
DeepMineRisk is an AI-powered mining safety monitoring system that predicts flotation process
conditions from sensor data and converts predictions into an intuitive 0-100 AI Risk Score with
Safe / Warning / Critical status and automatic recommendations.

## Objectives
- Predict key mining process/quality indicators using an LSTM deep learning model
- Translate raw predictions into an actionable Risk Score
- Provide automatic, risk-aware operational recommendations
- Deliver insights through an interactive Streamlit dashboard

## Dataset
Kaggle: MiningProcess_Flotation_Plant_Database.csv
Target column used: {TARGET_COL}

## Architecture
Input -> LSTM(32) -> Dropout(0.2) -> LSTM(16) -> Dense(16, relu) -> Dense(1)

## Workflow
1. Upload & extract dataset (ZIP -> CSV auto-detected)
2. Data cleaning (decimal-comma fix, dtype casting, missing values, duplicates)
3. Exploratory Data Analysis (Plotly visualizations)
4. Feature engineering (rolling stats, lag, moving average, difference features)
5. MinMax scaling (scaler persisted with pickle)
6. Sequence creation (sequence length = 10)
7. Train/test split (80/20, chronological)
8. LSTM training with EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
9. Evaluation (MAE, RMSE, R2, MAPE) + diagnostic plots
10. AI Risk Score (0-100) + Safe/Warning/Critical levels
11. Recommendation engine
12. Explainable AI with SHAP DeepExplainer (auto-skips if unsupported)
13. Streamlit dashboard (app.py) + deployment files

## Screenshots
_Add dashboard screenshots here after running `streamlit run app.py`._

## Installation
```bash
pip install -r requirements.txt
```

## How to Run
```bash
streamlit run app.py
```
Then open the local URL shown in the terminal, upload a sensor CSV, and click **Run Prediction**.

## Results
- MAE: {mae:.4f}
- RMSE: {rmse:.4f}
- R2 Score: {r2:.4f}
- MAPE: {mape:.2f}%

## Future Scope
- Integrate live sensor streaming (MQTT/Kafka) for real-time risk monitoring
- Extend to multi-target prediction (multiple quality indicators simultaneously)
- Deploy on cloud (AWS/GCP/Azure) with a scheduled retraining pipeline
- Add authentication and role-based dashboards for plant operators vs. supervisors
"""

with open(os.path.join(PROJECT_DIR, "README.md"), "w") as f:
    f.write(readme_md)

print("✅ README.md generated")

In [ ]:
gitignore_txt = """__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.DS_Store
*.log
venv/
.streamlit/
"""

with open(os.path.join(PROJECT_DIR, ".gitignore"), "w") as f:
    f.write(gitignore_txt)

print("✅ .gitignore generated")

## 20. Final Project Folder Structure

```
DeepMineRisk/
│
├── app.py
├── DeepMineRisk_Model.keras
├── scaler.pkl
├── features.pkl
├── requirements.txt
├── README.md
├── Prediction_Report.csv
├── .gitignore
├── dataset/
│   └── MiningProcess_Flotation_Plant_Database.csv
```

The cell below verifies every deliverable is present and zips the whole project folder for easy download.

In [ ]:
expected_files = [
    "app.py", "DeepMineRisk_Model.keras", "scaler.pkl", "features.pkl",
    "requirements.txt", "README.md", "Prediction_Report.csv", ".gitignore"
]

print("📁 Final Project Contents:\n")
for fname in expected_files:
    fpath = os.path.join(PROJECT_DIR, fname)
    status = "✅" if os.path.exists(fpath) else "❌"
    print(f"{status} {fname}")

# Copy the source CSV into the dataset/ folder with the standard name (if not already there)
target_csv = os.path.join(DATASET_DIR, "MiningProcess_Flotation_Plant_Database.csv")
if not os.path.exists(target_csv):
    shutil.copy(csv_path, target_csv)

# Zip the entire project folder for download
zip_output = "/content/DeepMineRisk_Project"
shutil.make_archive(zip_output, "zip", PROJECT_DIR)
print(f"\n✅ Project packaged: {zip_output}.zip")

## ✅ Project Complete

You now have a fully trained LSTM model, an AI risk-scoring pipeline, a recommendation engine, SHAP explainability, and a complete Streamlit dashboard with all deployment files — ready for your final-year AIML presentation and demonstration.

To launch the dashboard locally (or in Colab via `pyngrok`), run:
```bash
streamlit run /content/DeepMineRisk/app.py
```

In [ ]:
# ==============================
# Launch Streamlit with Cloudflare Tunnel
# ==============================

%cd /content/DeepMineRisk

# Kill any previous Streamlit processes
!pkill -f streamlit

# Install Streamlit if needed
!pip -q install streamlit

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start Streamlit in the background
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

import time
time.sleep(15)

# Start Cloudflare Tunnel
!./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &

time.sleep(10)

# Print the public URL
!grep -o 'https://[-0-9a-z]*\.trycloudflare.com' tunnel.log

In [ ]:
from google.colab import files

files.download("/content/DeepMineRisk/dataset/MiningProcess_Flotation_Plant_Database.csv")